In [3]:
import torch
import clip
from PIL import Image
import urllib.request
import pandas as pd
import numpy as np
import re
import requests
from io import BytesIO
import concurrent.futures
from tqdm import tqdm

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

In [5]:
df = pd.read_excel('../behaviour_simulation_test_time.xlsx')
df

,id,date,content,username,media,inferred company
0,1,2021-11-08 00:40:49,"Andres, a Pharmacy Manager at Store 4669 in Pi...",WalmartWorld,[Photo(previewUrl='https://pbs.twimg.com/media...,walmart
1,2,2021-05-09 07:51:21,Happy #EuropeDay! We are always stronger by wo...,SimonHarrisTD,[Photo(previewUrl='https://pbs.twimg.com/media...,td
2,3,2022-11-22 05:30:05,"Election Results, BVAS Report Established Over...",IndependentNGR,[Photo(previewUrl='https://pbs.twimg.com/media...,independent
3,4,2021-09-22 21:52:08,American Olympic swimmer Elizabeth Beisel will...,CBCOlympics,[Photo(previewUrl='https://pbs.twimg.com/media...,cbc
4,5,2022-04-15 13:00:27,No need to hunt ‘em or hope to find ‘em in a b...,TimHortonsUS,[Photo(previewUrl='https://pbs.twimg.com/media...,tim hortons
...,...,...,...,...,...,...
9995,9996,2021-04-27 11:28:28,𝐊𝐚𝐢𝐳𝐞𝐧 - The essence of continuous improvement...,TGR_WEC,[Video(thumbnailUrl='https://pbs.twimg.com/ext...,toyota
9996,9997,2021-10-10 10:08:29,JACK'S BACK! 🤩\n\nBest of luck to our Reserve ...,WilliamsRacing,[Photo(previewUrl='https://pbs.twimg.com/media...,williams
9997,9998,2022-10-15 05:13:00,[Special Saturday Hearing in #SupremeCourt]\n\...,barandbench,[Photo(previewUrl='https://pbs.twimg.com/media...,bar
9998,9999,2022-05-26 00:42:30,Obama has crossed the line. Don’t make this re...,w_terrence,[Photo(previewUrl='https://pbs.twimg.com/media...,williams


In [6]:
def clean(input):
    input_string = input
    
    full_url_match = re.search(r"fullUrl='([^']+)'", input_string)
    thumbnail_url_match = re.search(r"thumbnailUrl='([^']+)'", input_string)

    if full_url_match:
        full_url = full_url_match.group(1)
        return full_url
    elif thumbnail_url_match:
        thumbnail_url = thumbnail_url_match.group(1)
        return thumbnail_url

temp = []
for i in df['media']:
    temp.append(clean(i))
df['media'] = temp

In [7]:
df

,id,date,content,username,media,inferred company
0,1,2021-11-08 00:40:49,"Andres, a Pharmacy Manager at Store 4669 in Pi...",WalmartWorld,https://pbs.twimg.com/media/FDocI4fXsAcP-kl?fo...,walmart
1,2,2021-05-09 07:51:21,Happy #EuropeDay! We are always stronger by wo...,SimonHarrisTD,https://pbs.twimg.com/media/E07jvONWUAMNFER?fo...,td
2,3,2022-11-22 05:30:05,"Election Results, BVAS Report Established Over...",IndependentNGR,https://pbs.twimg.com/media/FiJQ5amaYAAzW0Y?fo...,independent
3,4,2021-09-22 21:52:08,American Olympic swimmer Elizabeth Beisel will...,CBCOlympics,https://pbs.twimg.com/media/E_68SlzVgAs0J5c?fo...,cbc
4,5,2022-04-15 13:00:27,No need to hunt ‘em or hope to find ‘em in a b...,TimHortonsUS,https://pbs.twimg.com/media/FQYwn-cWQAExho_?fo...,tim hortons
...,...,...,...,...,...,...
9995,9996,2021-04-27 11:28:28,𝐊𝐚𝐢𝐳𝐞𝐧 - The essence of continuous improvement...,TGR_WEC,https://pbs.twimg.com/ext_tw_video_thumb/13870...,toyota
9996,9997,2021-10-10 10:08:29,JACK'S BACK! 🤩\n\nBest of luck to our Reserve ...,WilliamsRacing,https://pbs.twimg.com/media/FBUsCk1XIAA4iEt?fo...,williams
9997,9998,2022-10-15 05:13:00,[Special Saturday Hearing in #SupremeCourt]\n\...,barandbench,https://pbs.twimg.com/media/FfFgkmNUAAAWMax?fo...,bar
9998,9999,2022-05-26 00:42:30,Obama has crossed the line. Don’t make this re...,w_terrence,https://pbs.twimg.com/media/FTpQ501WIAEXJpM?fo...,williams


In [8]:
def process_image_from_url(url, results, index):
    try:
        response = requests.get(url)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = model.encode_image(image)

        #print(f"Number of features for {url}: {len(image_features[0])}")

        results[index] = {
            'features': image_features[0].cpu().numpy().tolist(),
            'error': None
        }

    except Exception as e:
        #print(f"Error processing image at {url}: {e}")
        results[index] = {
            'features': None,
            'error': index
        }

#print("Error indices:", error_indices)
#print("Image embeddings:", image_embeddings)

In [9]:
image_urls = df['media']

with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    results = {}
    futures = [executor.submit(process_image_from_url, url, results, i) for i, url in enumerate(image_urls)]

    concurrent.futures.wait(futures)

error_indices = [index for index, result in results.items() if result['error'] is not None]
image_embeddings = [result['features'] for result in results.values() if result['features'] is not None]

print("Image processing completed.")

/home/bcca/anaconda3/envs/verb/lib/python3.9/site-packages/PIL/Image.py:975: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Image processing completed.


In [10]:
len(error_indices)

119

In [11]:
len(image_embeddings)

9881

In [12]:
dfie3 = pd.DataFrame(image_embeddings)
dfie3_err = pd.DataFrame(error_indices)
dfie3.to_csv("test_image_embeddings[time].csv")
dfie3_err.to_csv("test_image_embeddings_errors[time].csv")